# Lesson 01 · Watch Q-learning happen

We will play **full 4×4 2048**, follow one learning update, train a table of action
values, and check what it learned on new games.

Run cells in order. Every experiment writes to a new local folder. A fresh
kernel can run the entire notebook without previous outputs or checkpoints.
The default is a short 2,000-transition experiment; the full 100,000-transition
command appears near the end.

**Reading route:** game rules → random decisions → Q values → training loop →
evaluation → one-factor experiment. The source files are short enough to read
alongside this notebook.

In [ ]:
from dataclasses import asdict, replace
from datetime import datetime
import inspect
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

import rl2048
from rl2048.game import Game2048, ACTION_NAMES, merge_left
from rl2048.agents.random_agent import RandomAgent
from rl2048.agents.q_learning import QLearningAgent, state_key
from rl2048.train import TrainConfig, train
from rl2048.evaluate import evaluate, save_evaluation, rollout
from rl2048.view import board_html, play, replay_widget, save_replay, plot_training, plot_comparison

# Resolve the project from the imported package, not Jupyter's launch directory.
ROOT = Path(rl2048.__file__).resolve().parent.parent
RUN = ROOT / "runs" / ("lesson_01_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN.mkdir(parents=True)
print("Project:", ROOT)
print("This notebook's results:", RUN)

## 1. Play the environment

The board is a 4×4 NumPy array of **actual tile values** (`0` means empty).
The four actions are up, right, down, and left. A new tile appears only after a
move changes the board. Greyed-out controls are currently illegal moves.

Try predicting the result before clicking. `[2, 2, 4, 0]` moving left becomes
`[4, 4, 0, 0]`, with reward **4**. The newly made 4 cannot merge again in that move.

In [ ]:
play(seed=7)

In [ ]:
row, reward = merge_left(np.array([2, 2, 4, 0]))
print("After moving left:", row)
print("Immediate reward:", reward)

## 2. How the files connect

```text
                         lessons/01_q_learning.ipynb
                             /          |          \
                            v           v           v
                        train.py    evaluate.py    view.py
                         /   \        /    \      controls, plots, replays
                        v     v      v      v
                    game.py   agents/q_learning.py
                    rules     action selection + ONE learning update
                                (or agents/random_agent.py)

train.py  →  checkpoint.npz + episodes.csv + coverage.csv
checkpoint → evaluate.py → held-out results → view.py
```

The **environment** owns the board and spawning randomness. The **agent** owns
its Q table and action-selection randomness. The **training loop** passes data
between them. The **viewer** displays it. See the README's file map for every
source, test and configuration file.

Here is the complete interaction API. Notice that `info` includes legal moves.

In [ ]:
env = Game2048(render_mode="ansi")
board, info = env.reset(seed=7)
random_agent = RandomAgent(seed=7)
action = random_agent.act(board, info["action_mask"])
next_board, reward, terminated, truncated, next_info = env.step(action)
print("Action indices:", dict(enumerate(ACTION_NAMES)))
print("Chosen action:", ACTION_NAMES[action])
print("Reward:", reward, "terminated:", terminated, "truncated:", truncated)
print("Legal NEXT moves:", next_info["action_mask"])
print(env.render())
env.close()

## 3. What Q-learning stores

A **state** $s$ contains all 16 tiles in row order. Each distinct state has four
numbers $Q(s,a)$, one per action. They estimate the discounted future merge
reward after taking that action and then acting greedily.

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots$$

Here $r_{t+1}$ is the reward returned by the action at time $t$. Our code calls
this immediate reward simply `reward`. The raw game score sums undiscounted
rewards; it is different from this discounted objective when $\gamma < 1$.

Unseen states start with all zeros. The greedy action has the largest legal Q
value. With probability $\epsilon$, we explore by sampling a legal action
uniformly instead. Greedy ties are random too, preventing a directional bias.

In [ ]:
learner = QLearningAgent(alpha=0.1, gamma=0.99, seed=7)
print("State key:", state_key(board))
print("Q values:", dict(zip(ACTION_NAMES, learner.values(board))))
print("Legal-action mask:", info["action_mask"])
print("Exploratory action:", ACTION_NAMES[learner.act(board, info["action_mask"], epsilon=1.0)])
print("Table entries after reading an unseen state:", len(learner.q))

## 4. Follow one real update

$$y = r + \gamma\max_{a'\in A(s')} Q(s',a')$$
$$\delta = y-Q(s,a), \qquad Q(s,a) \leftarrow Q(s,a)+\alpha\delta$$

- $s$, `board`: the current board; $s'$, `next_board`: the returned next board.
- $a$, `action`: the action we took; $A(s')$: legal actions in the next board.
- $r$, `reward`: the immediate merge reward.
- $y$, `target`: the estimate we learn toward.
- $\delta$, `td_error`: the target minus our old estimate.
- $\alpha$, `alpha`: learning rate, here 0.1; $\gamma$, `gamma`: discount, here 0.99.

At natural termination, `next_value = 0`, so the target is just the reward.
A time limit is **truncation**: we stop collecting this episode, but the board
still has future value. We bootstrap using that final board before resetting.

The next cell collects real transitions until the first merge, updating the
actual agent on each transition, and exposes every number from that update.

In [ ]:
env = Game2048()
board, info = env.reset(seed=7)
trace_agent = QLearningAgent(alpha=0.1, gamma=0.99, seed=7)
for t in range(1000):
    action = trace_agent.act(board, info["action_mask"], epsilon=1.0)
    next_board, reward, terminated, truncated, next_info = env.step(action)
    trace = trace_agent.update(board, action, reward, next_board, terminated, next_info["action_mask"])
    if reward > 0:
        break
    if terminated or truncated:
        board, info = env.reset()
    else:
        board, info = next_board, next_info
assert reward > 0
print("Transition number:", t + 1, "| action:", ACTION_NAMES[action])
display(HTML(board_html(board, caption="Before action")))
display(HTML(board_html(next_board, next_info["score"], "After action and spawn")))
for name, value in asdict(trace).items():
    print(f"{name:>12}: {value:.4f}")
print("Updated row:", trace_agent.values(board))
env.close()

The next state is probably new, so its value is zero. That makes the first
positive update especially simple: with reward 4, alpha 0.1, and old value 0,
the new value is 0.4.

To see bootstrapping clearly, the next cell uses an **explicitly constructed
arithmetic example**, not learned estimates. Illegal next actions have very
large values; they must still be excluded from the target.

In [ ]:
example = QLearningAgent(alpha=0.1, gamma=0.99)
s = np.zeros((4, 4), dtype=np.int64)
s[0, :2] = [2, 2]
s_next = np.zeros_like(s)
s_next[0, 0] = 4
example.q[state_key(s)] = np.array([0., 2., 0., 0.])
example.q[state_key(s_next)] = np.array([1000., 5., 10., 2000.])
example_trace = example.update(s, 1, 4, s_next, False, np.array([False, True, True, False]))
print(asdict(example_trace))
assert np.isclose(example_trace.target, 13.9)
assert np.isclose(example_trace.new_value, 3.19)

## 5. Read the actual training loop, then run it

Read `train()` below from top to bottom. Its four central steps are **act →
step → update → advance/reset**. The surrounding code saves configuration,
metrics and the checkpoint. `config.steps` counts environment transitions,
not games and not repeated optimization passes.

This is **off-policy**: experience comes from an epsilon-greedy behavior policy,
while the target uses the greedy legal next action. We do not need to actually
take that next greedy action to form this Q-learning target.

In [ ]:
print(inspect.getsource(train))

In [ ]:
config = TrainConfig(steps=2000, seed=0)
agent, training_summary = train(config, RUN / "q_learning")
print(json.dumps(training_summary, indent=2))

## 6. Inspect learning and coverage

The score curve contains **completed training games** under an exploratory,
changing policy. It is not a frozen-policy evaluation. The middle plot compares
the number of distinct updated boards with the number of updates. If those
curves overlap, almost every update was on a board we had never learned on.

Epsilon stays near 1 during a 2,000-transition smoke run because we retain the
full experiment's 100,000-transition decay schedule.

In [ ]:
figure = plot_training(RUN / "q_learning")
figure.savefig(RUN / "training.png", dpi=160)
plt.show()
coverage = agent.coverage()
print(f"Distinct boards: {coverage['unique_states']:,}")
print(f"Repeated updates: {coverage['repeated_updates']:,}")
print(f"Fraction of updates revisiting a board: {coverage['repeated_update_fraction']:.2%}")

## 7. Evaluate a saved, frozen agent

Load the checkpoint to verify we are evaluating a reusable result. Both agents
play the same **100 held-out game seeds**. Q-learning uses epsilon zero; it still
breaks ties randomly. Neither evaluation inserts states or updates Q values.

These games are independent of the training RNG stream. Matching seeds does not
force the same complete spawn history when agents take different actions.

In [ ]:
frozen = QLearningAgent.load(RUN / "q_learning" / "checkpoint.npz")
coverage_before = frozen.coverage()
random_summary = save_evaluation(RandomAgent(), RUN / "random_eval")
q_summary = save_evaluation(frozen, RUN / "q_eval", checkpoint=str(RUN / "q_learning/checkpoint.npz"))
assert frozen.coverage() == coverage_before

def comparison_table(summaries):
    rows = ["| Agent | Mean score | Median score | Score std | Mean moves | >=256 | >=512 |",
            "|---|---:|---:|---:|---:|---:|---:|"]
    for name, s in summaries.items():
        rows.append(f"| {name} | {s['mean_score']:.1f} | {s['median_score']:.1f} | {s['score_std']:.1f} | {s['mean_episode_length']:.1f} | {s['tile_reaching_rates']['256']:.0%} | {s['tile_reaching_rates']['512']:.0%} |")
    display(Markdown("\n".join(rows)))

comparison_table({"Random": random_summary, "Q-learning": q_summary})
print(f"Evaluated boards already known to Q-learning: {q_summary['known_state_fraction']:.2%}")
print("Maximum-tile distribution:", q_summary["max_tile_distribution"])
print("Evaluation transitions:", q_summary["environment_transitions"])
print("Evaluation seconds:", round(q_summary["elapsed_seconds"], 3))

In [ ]:
figure = plot_comparison({"Random": random_summary, "Q-learning": q_summary})
figure.savefig(RUN / "evaluation.png", dpi=160)
plt.show()

## 8. Watch a complete recorded game

This replay uses a separate demonstration seed. It is a single example, not a
performance estimate. Scrub through the moves or press Play. We also save a
standalone HTML viewer that works outside Jupyter, and its raw JSON frames.

In [ ]:
replay_result = save_replay(frozen, RUN / "replay.html")
replay_data = json.loads((RUN / "replay.json").read_text())
print("Saved viewer:", RUN / "replay.html")
print("Final result:", replay_result)
replay_widget(replay_data["frames"])

## 9. Change one factor: how much does the future matter yet?

Keep the same seed, 2,000 transitions, alpha, and exploration schedule. Change
only $\gamma$ from 0.99 to 0. With $\gamma=0$, the target is just the immediate
reward. Does that change the learned table or evaluation in this short run?

Predict the result first. Even with $\gamma>0$, a zero next-state estimate makes
the bootstrap term zero. Sparse state reuse can hide the effect of discounting.

In [ ]:
myopic, myopic_training = train(replace(config, gamma=0.0), RUN / "gamma_zero")
myopic_summary = save_evaluation(myopic, RUN / "gamma_zero_eval", checkpoint=str(RUN / "gamma_zero/checkpoint.npz"))
comparison_table({"Random": random_summary, "gamma=0.99": q_summary, "gamma=0": myopic_summary})
shared_states = set(agent.q) & set(myopic.q)
changed_rows = sum(not np.array_equal(agent.q[key], myopic.q[key]) for key in shared_states)
print(f"Shared states: {len(shared_states):,}; rows with different values: {changed_rows:,}")
if set(agent.q) == set(myopic.q) and changed_rows == 0:
    display(Markdown("**Observed:** the tables are identical in this run. The future-value term did not distinguish these updates. Inspect the near-zero state reuse before interpreting this as a result about gamma."))
else:
    display(Markdown("**Observed:** the runs differ. Changing gamma can change targets on known next states, then alter future greedy actions and the states collected. One short training seed does not establish which setting is better."))

## 10. The full first experiment

Run this from the project root in a terminal to collect 100,000 transitions:

```bash
uv run python -m rl2048.train --out runs/q_learning
uv run python -m rl2048.evaluate --checkpoint runs/q_learning/checkpoint.npz --out runs/q_eval
uv run python -m rl2048.view --checkpoint runs/q_learning/checkpoint.npz --out runs/q_learning/replay.html --open
```

The cell below reads that run **if it already exists**; it does not launch a long
training job or overwrite results when you run this notebook again.

In [ ]:
full_run = ROOT / "runs" / "q_learning"
if (full_run / "summary.json").exists():
    print((full_run / "summary.json").read_text())
    plot_training(full_run)
    plt.show()
else:
    print("No full run saved yet. Use the terminal command above when ready.")

## What to ask about next

- Why does one update change only one number in one board's row?
- How can Q-learning be off-policy while acting epsilon-greedily?
- Why does a time limit still bootstrap, while a lost game does not?
- Why can a larger Q table fail to produce better play?
- How would SARSA's target differ on the same transition?

**Next lesson:** Monte Carlo control, SARSA, and Expected SARSA. Later we will
introduce DQN to share learning across boards, then REINFORCE, actor–critic,
GAE and PPO. We will build and explain each stage separately.